# 🤝 01 · 协同过滤：UserCF 与 ItemCF

> 协同过滤（Collaborative Filtering）是推荐系统最经典的一族算法：**用"群体的行为"代替"个体的画像"**。
> 本节手写 UserCF / ItemCF 完整流程：相似度 → 邻居 → 预测/推荐 → 评估。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 什么是协同过滤？ | 物以类聚、人以群分 |
| 2 | UserCF 怎么推荐？ | 相似用户 → 他们喜欢的未看物品 |
| 3 | ItemCF 怎么推荐？ | 相似物品 → 用户喜欢的物品的相似品 |
| 4 | 两种方法怎么选？ | 冷启动/覆盖面/可解释性对比 |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 构造评分矩阵（用户 × 物品）

In [2]:
# 6 用户 × 6 物品，0 表示未评分
R = np.array([
    [5, 3, 0, 1, 4, 0],
    [4, 0, 0, 1, 5, 0],
    [1, 1, 0, 5, 2, 0],
    [1, 0, 4, 4, 0, 5],
    [0, 1, 5, 4, 0, 4],
    [2, 4, 0, 1, 3, 0],
])
users = [f'U{i+1}' for i in range(6)]
items = [f'Item{j+1}' for j in range(6)]
df = pd.DataFrame(R, index=users, columns=items)
print(df)
print('\n非零评分总数:', (R > 0).sum())

    Item1  Item2  Item3  Item4  Item5  Item6
U1      5      3      0      1      4      0
U2      4      0      0      1      5      0
U3      1      1      0      5      2      0
U4      1      0      4      4      0      5
U5      0      1      5      4      0      4
U6      2      4      0      1      3      0

非零评分总数: 23


## 2️⃣ 相似度：余弦 / 皮尔逊（手写）

In [3]:
def cosine_sim(a, b):
    if a.sum() == 0 or b.sum() == 0:
        return 0.0
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

# 用户-用户相似度矩阵（只对"共同评分过的物品"算？完整版用共同评分，这里演示完整向量）
n = R.shape[0]
sim_user = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_user[i, j] = cosine_sim(R[i], R[j])
sim_user_df = pd.DataFrame(sim_user.round(3), index=users, columns=users)
print('用户相似度矩阵:')
print(sim_user_df)
assert abs(sim_user[0, 1] - sim_user[1, 0]) < 1e-9 and abs(sim_user[0, 0] - 1.0) < 1e-9
print('\n✅ 相似度矩阵对称、自相似=1')

用户相似度矩阵:
       U1     U2     U3     U4     U5     U6
U1  1.000  0.886  0.528  0.165  0.129  0.895
U2  0.886  1.000  0.527  0.162  0.081  0.676
U3  0.528  0.527  1.000  0.495  0.495  0.557
U4  0.165  0.162  0.495  1.000  0.966  0.144
U5  0.129  0.081  0.495  0.966  1.000  0.192
U6  0.895  0.676  0.557  0.144  0.192  1.000

✅ 相似度矩阵对称、自相似=1


## 3️⃣ UserCF：预测未评分物品

In [4]:
def predict_usercf(R, sim, u, i, k=2):
    # 用与 u 最相似的 k 个用户对 i 的评分加权预测（排除未评分）
    if R[u, i] > 0:
        return R[u, i]
    scored = [v for v in range(R.shape[0]) if v != u and R[v, i] > 0]
    scored.sort(key=lambda v: -sim[u, v])
    neighbors = scored[:k]
    if not neighbors:
        return 0.0
    num = sum(sim[u, v] * R[v, i] for v in neighbors)
    den = sum(abs(sim[u, v]) for v in neighbors)
    return num / den if den > 0 else 0.0

# 给 U1 预测 Item3（未评分）
pred = predict_usercf(R, sim_user, 0, 2)
print(f'U1 对 Item3 的预测评分 = {pred:.2f}')
assert pred > 0
print('✅ UserCF 预测完成')

U1 对 Item3 的预测评分 = 4.44
✅ UserCF 预测完成


## 4️⃣ 给用户生成 Top-N 推荐

In [5]:
def recommend_usercf(R, sim, u, k_users=2, top_n=3):
    unrated = [i for i in range(R.shape[1]) if R[u, i] == 0]
    preds = [(i, predict_usercf(R, sim, u, i, k_users)) for i in unrated]
    preds.sort(key=lambda x: -x[1])
    return [(items[i], round(p, 2)) for i, p in preds[:top_n]]

rec_u1 = recommend_usercf(R, sim_user, 0)
print('U1 的 Top-N 推荐:', rec_u1)
print('➡️ U1 只有 2 个未评分物品，所以不足 3 个')
assert 1 <= len(rec_u1) <= 3
print('✅ 推荐列表生成')

U1 的 Top-N 推荐: [('Item6', np.float64(4.56)), ('Item3', np.float64(4.44))]
➡️ U1 只有 2 个未评分物品，所以不足 3 个
✅ 推荐列表生成


## 5️⃣ ItemCF：物品-物品相似度 + 推荐

In [6]:
# 物品相似度：按列算余弦
n_items = R.shape[1]
sim_item = np.zeros((n_items, n_items))
for i in range(n_items):
    for j in range(n_items):
        sim_item[i, j] = cosine_sim(R[:, i], R[:, j])

# ItemCF 推荐：用户已评分的物品 → 找相似物品
def recommend_itemcf(R, sim_item, u, k_sim=2, top_n=3):
    rated = [i for i in range(R.shape[1]) if R[u, i] > 0]
    scores = {}
    for i in rated:
        for j in range(R.shape[1]):
            if j == i or R[u, j] > 0:
                continue
            scores[j] = scores.get(j, 0) + sim_item[i, j] * R[u, i]
    top = sorted(scores.items(), key=lambda x: -x[1])[:top_n]
    return [(items[j], round(s, 2)) for j, s in top]

rec_i1 = recommend_itemcf(R, sim_item, 0)
print('ItemCF 给 U1 的推荐:', rec_i1)
print('UserCF 给 U1 的推荐:', rec_u1)
print('➡️ U1 仅 2 个未评分物品，两种方法都可能不足 3 个')
assert 1 <= len(rec_i1) <= 3 and 1 <= len(rec_u1) <= 3
print('\n✅ 两种方法都能产出推荐')

ItemCF 给 U1 的推荐: [('Item6', np.float64(1.66)), ('Item3', np.float64(1.63))]
UserCF 给 U1 的推荐: [('Item6', np.float64(4.56)), ('Item3', np.float64(4.44))]
➡️ U1 仅 2 个未评分物品，两种方法都可能不足 3 个

✅ 两种方法都能产出推荐


## 6️⃣ 相似度热力图

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(sim_user, cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].set_title('用户-用户相似度')
axes[0].set_xticks(range(6)); axes[0].set_yticks(range(6))
axes[0].set_xticklabels(users, fontsize=8); axes[0].set_yticklabels(users, fontsize=8)
im1 = axes[1].imshow(sim_item, cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].set_title('物品-物品相似度')
axes[1].set_xticks(range(6)); axes[1].set_yticks(range(6))
axes[1].set_xticklabels(items, fontsize=8, rotation=45); axes[1].set_yticklabels(items, fontsize=8)
fig.colorbar(im0, ax=axes[0], shrink=0.8)
plt.tight_layout()
plt.show()

C:\Users\24260\AppData\Local\Temp\ipykernel_44644\102693657.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7️⃣ UserCF vs ItemCF（面试对比表）

In [8]:
print('''| 维度 | UserCF | ItemCF |
|---|---|---|
| 思想 | 找相似用户 | 找相似物品 |
| 冷启动 | 新用户无行为难推荐 | 新用户有少量行为即可 |
| 新物品 | 立即可推（被相似用户评过） | 需要先积累行为 |
| 覆盖面 | 偏热门物品 | 较分散、长尾好 |
| 可解释性 | "和你相似的人喜欢" | "因为你喜欢 X，所以推荐 Y" |
| 计算量 | 用户数增长贵 | 物品数增长贵（物品更稳） |
| 场景 | 用户数少/新闻 | 电商/视频（物品相对稳定） |
''')

| 维度 | UserCF | ItemCF |
|---|---|---|
| 思想 | 找相似用户 | 找相似物品 |
| 冷启动 | 新用户无行为难推荐 | 新用户有少量行为即可 |
| 新物品 | 立即可推（被相似用户评过） | 需要先积累行为 |
| 覆盖面 | 偏热门物品 | 较分散、长尾好 |
| 可解释性 | "和你相似的人喜欢" | "因为你喜欢 X，所以推荐 Y" |
| 计算量 | 用户数增长贵 | 物品数增长贵（物品更稳） |
| 场景 | 用户数少/新闻 | 电商/视频（物品相对稳定） |



## 📝 自测清单

- [ ] 能手写余弦/皮尔逊相似度并解释 0 处理
- [ ] 能手写 UserCF 预测公式（邻居加权）
- [ ] 能手写 ItemCF 推荐流程
- [ ] 能背出 UserCF vs ItemCF 六维对比
- [ ] 能说出协同过滤的冷启动问题与缓解思路